## Rading from files

In [0]:
import pyspark.pandas as ps
df=ps.read_csv("/Volumes/workspace/bronze/data_sources/source_crm/cust_info.csv",index_col="cst_id")


## Write Into Bronze

In [0]:
df.to_table(name='workspace.bronze.cust_info',format='delta',mode='overwrite')

In [0]:
# 1. تحديد مسارات الفولدرات التي تحتوي على مصادر البيانات المختلفة
sources = [
    "/Volumes/workspace/bronze/data_sources/source_crm/",
    "/Volumes/workspace/bronze/data_sources/source_erp/"
]

# 2. الحلقة التكرارية الكبرى على الفولدرات
for folder_path in sources:
    print(f"📁 بدأ العمل على الفولدر: {folder_path}\n" + "="*50)
    
    # الحصول على قائمة الملفات داخل الفولدر الحالي
    try:
        files = dbutils.fs.ls(folder_path)
    except Exception as e:
        print(f"⚠️ يتعذر الوصول للمسار: {folder_path}. يرجى التأكد من صحة المسار.\n")
        continue

    # 3. الحلقة التكرارية على كل ملف داخل الفولدر
    for file in files:
        if file.name.endswith('.csv'):
            # تجهيز اسم الجدول عن طريق إزالة امتداد .csv 
            # مثال: cust_info.csv تصبح crm_cust_info أو تظل كما هي بحسب تفضيلك
            table_name = file.name.replace('.csv', '').lower()
            
            # تحديد اسم الجدول الكامل في الكاتالوج
            full_table_name = f"workspace.bronze.{table_name}"
            
            print(f"⏳ جاري قراءة الملف: {file.name} ...")
            
            # قراءة ملف CSV وتحديد أنواع البيانات تلقائياً
            df = spark.read \
                .option("header", "true") \
                .option("inferSchema", "true") \
                .csv(file.path)
            
            print(f"💾 جاري الحفظ كـ Delta Table باسم: {full_table_name} ...")
            
            # كتابة البيانات كـ Delta Table واستبدال القديم إن وجد
            df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(full_table_name)
            
            print(f"✅ تم إنشاء الجدول {full_table_name} بنجاح!\n" + "-"*30)

print("🎉 اكتملت عملية رفع جميع الملفات إلى طبقة الـ Bronze بنجاح!")